# Data process output for Taylor Diagram & Wasserstein-Taylor Diagram

- Load different models (MLP, EBM, Lr, etc.) prediction on convective cloud fractions.
- Load ground truth (CONUS404) convective cloud fraction.
- Output (one-day) flattened data for Taylor diagram analysis.

**Hungjui – 20260529**


In [1]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [2]:
import time
import datetime as dt

import numpy as np
import xarray as xr
import pandas as pd

***
## Load saved prediction datasets:

In [3]:
## Load multiple prediction datasets:

sel_mode = 'freq_Conv'
# sel_mode = "freq_DWCC_mod"

wrf_sim_type = 'CTRL'

ana_year = '2022'
ana_yearmonth_range = ['01', '09']


path_prediction_data_1 = f'/glade/derecho/scratch/hungjui/DATA_prediction_NN_CoarseRes/'
path_prediction_data_2 = f'/glade/derecho/scratch/hungjui/DATA_prediction_EBM_CoarseRes/'
path_prediction_data_3 = f'/glade/derecho/scratch/hungjui/DATA_prediction_EBM_CoarseRes/regional_models/'
path_prediction_data_4 = f'/glade/derecho/scratch/hungjui/DATA_prediction_LR_CoarseRes'


prediction_1_name = f'021_NN_dur_2018_2021_freq_Conv_var16_rmAll90.0'
prediction_2_name = f'MergedEBM_2018_2021_freq_Conv_var16_rmAll90.0'
prediction_3_name = f'EBM_dur_2018_2021_freq_Conv_var16_rmAll90.0_MergedRegions'
prediction_4_name = f'LR_dur_2018_2021_freq_Conv_var16_rmAll90.0'

ds_prediction_1 = xr.open_dataset(f'{path_prediction_data_1}/{prediction_1_name}_prediction_{wrf_sim_type}_{sel_mode}_{ana_year}.nc') 
ds_prediction_2 = xr.open_dataset(f'{path_prediction_data_2}/{prediction_2_name}_prediction_{wrf_sim_type}_{sel_mode}_{ana_year}.nc') 
ds_prediction_3 = xr.open_dataset(f'{path_prediction_data_3}/{prediction_3_name}_prediction_{wrf_sim_type}_{sel_mode}_{ana_year}.nc') 
ds_prediction_4 = xr.open_dataset(f'{path_prediction_data_4}/{prediction_4_name}_prediction_{wrf_sim_type}_{sel_mode}_{ana_year}.nc') 

# da_prediction_1_avg = ds_prediction_1[f'predicted_{sel_mode}'].mean(dim='time') #.sel(time=slice('2022-06-01', '2022-09-30')).mean(dim='time')
# da_prediction_2_avg = ds_prediction_2[f'predicted_{sel_mode}'].mean(dim='time') #.sel(time=slice('2022-06-01', '2022-09-30')).mean(dim='time')
# da_prediction_3_avg = ds_prediction_3[f'predicted_{sel_mode}'].mean(dim='time') #.sel(time=slice('2022-06-01', '2022-09-30')).mean(dim='time')
# da_prediction_4_avg = ds_prediction_4[f'predicted_{sel_mode}'].mean(dim='time') #.sel(time=slice('2022-06-01', '2022-09-30')).mean(dim='time')



In [ ]:
ds_prediction_4

***
## Load ground truth:

In [4]:
ds_ground_truth = xr.open_dataset(f'/glade/derecho/scratch/hungjui/DATA_WRF_CONUS_404_SMode_v1.0/StormMode_Freq_CoarseRes_CTRL/wrf3d_404_{wrf_sim_type[0]}_freq_csmode_{ana_year}{ana_yearmonth_range[0]}_{ana_year}{ana_yearmonth_range[-1]}.nc')

# da_ground_truth = ds_ground_truth[sel_mode]
# da_ground_truth

***
## Flatten and Output:

In [5]:
# specific_time = np.datetime64('2022-03-22T00:00:00')
specific_time = np.datetime64('2022-08-22T02:00:00')
# specific_time = np.datetime64('2022-05-13T00:00:00')
# specific_time = np.datetime64('2022-04-10T15:00:00')

specific_time_str = specific_time.astype(dt.datetime).strftime('%Y%m%d%H')

print(specific_time_str)

2022082202


In [6]:
da_ground_truth = ds_ground_truth[sel_mode].sel(time=specific_time, method='nearest')

da_prediction_1 = ds_prediction_1[f'predicted_{sel_mode}'].sel(time=specific_time, method='nearest')
da_prediction_2 = ds_prediction_2[f'predicted_{sel_mode}'].sel(time=specific_time, method='nearest')
da_prediction_3 = ds_prediction_3[f'predicted_{sel_mode}'].sel(time=specific_time, method='nearest')
da_prediction_4 = ds_prediction_4[f'predicted_{sel_mode}'].sel(time=specific_time, method='nearest')

da_ground_truth #.plot()

<xarray.DataArray 'freq_Conv' (south_north: 145, west_east: 195)> Size: 226kB
[28275 values with dtype=float64]
Coordinates:
    time     datetime64[ns] 8B 2022-08-22T02:00:00
    XLAT     (south_north, west_east) float32 113kB ...
    XLONG    (south_north, west_east) float32 113kB ...
Dimensions without coordinates: south_north, west_east

In [9]:
XLAT_flat = da_ground_truth.XLAT.values.ravel()
XLONG_flat = da_ground_truth.XLONG.values.ravel()

df_flat = pd.DataFrame({ 'XLAT': XLAT_flat, 'XLONG': XLONG_flat })

df_flat['CONUS404'] = da_ground_truth.values.ravel()
df_flat['MLP'] = da_prediction_1.values.ravel()
df_flat['EBM'] = da_prediction_2.values.ravel()
df_flat['EBMregion'] = da_prediction_3.values.ravel()
df_flat['LR'] = da_prediction_3.values.ravel()

df_flat

,XLAT,XLONG,CONUS404,MLP,EBM,EBMregion,LR
0,17.773369,-122.500519,0.0,0.159576,-0.379876,0.037279,0.037279
1,17.838280,-122.259888,0.0,0.171610,-0.351258,0.094101,0.094101
2,17.902573,-122.018890,0.0,0.038266,-1.333828,0.085094,0.085094
3,17.966286,-121.777527,0.0,-0.048503,-1.521450,0.005902,0.005902
4,18.029396,-121.535797,0.0,-0.119472,-1.177949,0.075104,0.075104
...,...,...,...,...,...,...,...
28270,52.110783,-58.864868,0.0,0.429954,0.449916,0.620277,0.620277
28271,52.003799,-58.497437,0.0,0.591386,0.609142,1.208560,1.208560
28272,51.895836,-58.131470,0.0,0.588447,0.766448,0.259749,0.259749
28273,51.786900,-57.766937,0.0,0.294763,-0.397502,-1.013907,-1.013907


In [11]:
df_flat.to_parquet('./Data/caig_model_eval.parquet', index=False)